# ISIC 2024 strict_input export audit

이 노트북은 `export_strict_input_dataset` CLI가 만든 산출물을 검토하는 audit report다. 산출물을 생성하지 않고, 이미 생성된 `data/processed/`, `data/splits/`, `experiments/evidence/validation_protocol/` 파일을 읽어서 patient-level split, `iddx_full` 분리, fold 분포를 확인한다.

Source of truth 생성 명령:

```bash
PYTHONPATH=./src python -m isic2024_multimodal.cli.export_strict_input_dataset
```


## 1. Audit scope

- `isic2024_strict_model_input.csv`는 ordinary inference-time `strict_input` feature table이다.
- `isic2024_iddx_full_train_only_sidecar.csv`는 candidate-only privileged supervision sidecar다.
- `iddx_full`과 diagnosis/reference fields는 strict model input에 없어야 한다.
- `train_validation_data`와 `test_data`, 그리고 각 CV fold의 `cv_train`과 `cv_validation`은 patient-disjoint여야 한다.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'isic2024_multimodal').exists():
            return candidate
    raise RuntimeError('Could not locate repository root from current working directory')

REPO_ROOT = find_repo_root()
STRICT_INPUT_PATH = REPO_ROOT / 'data/processed/isic2024_strict_model_input.csv'
IDDX_SIDECAR_PATH = REPO_ROOT / 'data/processed/isic2024_iddx_full_train_only_sidecar.csv'
HOLDOUT_SPLIT_PATH = REPO_ROOT / 'data/splits/isic2024_train_validation_test_split_seed42.csv'
CV_SPLIT_PATH = REPO_ROOT / 'data/splits/isic2024_train_validation_5fold_seed42.csv'
SUMMARY_PATH = REPO_ROOT / 'experiments/evidence/validation_protocol/isic2024_strict_input_export_summary_seed42.json'

paths = {
    'strict_input': STRICT_INPUT_PATH,
    'iddx_sidecar': IDDX_SIDECAR_PATH,
    'holdout_split': HOLDOUT_SPLIT_PATH,
    'cv_split': CV_SPLIT_PATH,
    'summary': SUMMARY_PATH,
}

path_audit_df = pd.DataFrame(
    [{'artifact': name, 'path': path.as_posix(), 'exists': path.exists()} for name, path in paths.items()]
)
display(Markdown('**표 1. export artifact existence**'))
display(path_audit_df)

missing = path_audit_df.loc[~path_audit_df['exists'], 'path'].tolist()
if missing:
    raise FileNotFoundError('Run the export CLI before this audit notebook. Missing: ' + ', '.join(missing))


## 2. Load artifacts

이 셀은 CLI 산출물을 읽기만 한다. raw data나 processed data를 수정하지 않는다.


In [ ]:
strict_df = pd.read_csv(STRICT_INPUT_PATH, low_memory=False)
iddx_sidecar_df = pd.read_csv(IDDX_SIDECAR_PATH, low_memory=False)
holdout_split_df = pd.read_csv(HOLDOUT_SPLIT_PATH, low_memory=False)
cv_split_df = pd.read_csv(CV_SPLIT_PATH, low_memory=False)
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

artifact_shape_df = pd.DataFrame(
    [
        {'artifact': 'strict_input', 'rows': len(strict_df), 'columns': strict_df.shape[1]},
        {'artifact': 'iddx_sidecar', 'rows': len(iddx_sidecar_df), 'columns': iddx_sidecar_df.shape[1]},
        {'artifact': 'holdout_split', 'rows': len(holdout_split_df), 'columns': holdout_split_df.shape[1]},
        {'artifact': 'cv_split', 'rows': len(cv_split_df), 'columns': cv_split_df.shape[1]},
    ]
)
display(Markdown('**표 2. artifact shape**'))
display(artifact_shape_df)


## 3. Input leakage audit

Strict input table은 inference-time ordinary metadata만 포함해야 한다. `iddx_full`, `iddx_1`~`iddx_5`, `mel_*`, `tbp_lv_dnn_lesion_confidence`, provenance/context 컬럼은 main input에 들어가면 안 된다.


In [ ]:
disallowed_main_columns = {
    'iddx_full', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5',
    'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence',
    'attribution', 'copyright_license', 'image_type',
}
input_leakage_audit_df = pd.DataFrame(
    [
        {'check': 'strict_input_has_isic_id_unique', 'pass': bool(strict_df['isic_id'].is_unique), 'value': strict_df['isic_id'].nunique()},
        {'check': 'sidecar_aligned_by_isic_id', 'pass': bool(strict_df['isic_id'].equals(iddx_sidecar_df['isic_id'])), 'value': len(iddx_sidecar_df)},
        {'check': 'iddx_full_excluded_from_strict_input', 'pass': 'iddx_full' not in strict_df.columns, 'value': 'iddx_full' in strict_df.columns},
        {'check': 'diagnosis_reference_columns_excluded', 'pass': not bool(set(strict_df.columns) & disallowed_main_columns), 'value': sorted(set(strict_df.columns) & disallowed_main_columns)},
        {'check': 'sidecar_has_iddx_full_train_only', 'pass': 'iddx_full_train_only' in iddx_sidecar_df.columns, 'value': 'iddx_full_train_only' in iddx_sidecar_df.columns},
    ]
)
display(Markdown('**표 3. strict input / iddx sidecar leakage audit**'))
display(input_leakage_audit_df)
if not input_leakage_audit_df['pass'].all():
    raise AssertionError('Input leakage audit failed')


## 4. Patient-level split audit

Holdout split은 `patient_id` 기준으로 `train_validation_data`와 `test_data`를 분리해야 한다. CV split은 `train_validation_data` 내부에서만 fold를 만든다.


In [ ]:
strict_with_holdout_df = strict_df.merge(holdout_split_df[['isic_id', 'split']], on='isic_id', how='inner', validate='1:1')
holdout_summary_df = (
    strict_with_holdout_df.groupby('split', dropna=False)
    .agg(rows=('isic_id', 'size'), patients=('patient_id', 'nunique'), positive_rows=('target', 'sum'))
    .reset_index()
)
holdout_summary_df['positive_rate_pct'] = (holdout_summary_df['positive_rows'] / holdout_summary_df['rows'] * 100).round(5)

train_validation_patients = set(holdout_split_df.loc[holdout_split_df['split'].eq('train_validation_data'), 'patient_id'].astype(str))
test_patients = set(holdout_split_df.loc[holdout_split_df['split'].eq('test_data'), 'patient_id'].astype(str))
holdout_overlap_df = pd.DataFrame(
    [{'check': 'train_validation_test_patient_overlap', 'value': len(train_validation_patients & test_patients), 'pass': len(train_validation_patients & test_patients) == 0}]
)

display(Markdown('**표 4-a. holdout split summary**'))
display(holdout_summary_df)
display(Markdown('**표 4-b. holdout patient overlap**'))
display(holdout_overlap_df)
if not holdout_overlap_df['pass'].all():
    raise AssertionError('Holdout patient overlap audit failed')


In [ ]:
strict_with_cv_df = strict_df.merge(cv_split_df[['isic_id', 'cv_validation_fold']], on='isic_id', how='inner', validate='1:1')
cv_summary_df = (
    strict_with_cv_df.groupby('cv_validation_fold', dropna=False)
    .agg(rows=('isic_id', 'size'), patients=('patient_id', 'nunique'), positive_rows=('target', 'sum'))
    .reset_index()
    .sort_values('cv_validation_fold')
)
cv_summary_df['positive_rate_pct'] = (cv_summary_df['positive_rows'] / cv_summary_df['rows'] * 100).round(5)

all_cv_patients = set(cv_split_df['patient_id'].astype(str))
cv_overlap_rows = []
for fold in sorted(cv_split_df['cv_validation_fold'].unique()):
    validation_patients = set(cv_split_df.loc[cv_split_df['cv_validation_fold'].eq(fold), 'patient_id'].astype(str))
    train_patients = all_cv_patients - validation_patients
    cv_overlap_rows.append(
        {
            'cv_fold': int(fold),
            'cv_train_cv_validation_patient_overlap': len(train_patients & validation_patients),
            'cv_validation_test_data_patient_overlap': len(validation_patients & test_patients),
            'cv_train_test_data_patient_overlap': len(train_patients & test_patients),
        }
    )
cv_overlap_df = pd.DataFrame(cv_overlap_rows)

display(Markdown('**표 5-a. cv validation fold summary**'))
display(cv_summary_df)
display(Markdown('**표 5-b. cv patient overlap audit**'))
display(cv_overlap_df)
if not (cv_overlap_df.drop(columns=['cv_fold']) == 0).all().all():
    raise AssertionError('CV patient overlap audit failed')


## 5. Summary evidence

CLI가 저장한 JSON summary는 실행 seed, split score, leakage controls, artifact path를 기록한다. 이 정보는 결과표에서 split version과 threshold source를 추적할 때 사용한다.


In [ ]:
summary_overview_df = pd.DataFrame(
    [
        {'item': 'seed', 'value': summary['seed']},
        {'item': 'test_size', 'value': summary['test_size']},
        {'item': 'cv_folds', 'value': summary['cv_folds']},
        {'item': 'strict_input_column_count', 'value': summary['strict_input_contract']['strict_input_column_count']},
        {'item': 'holdout_balance_score', 'value': summary['split_scores']['holdout_balance_score']},
        {'item': 'cv_balance_score', 'value': summary['split_scores']['cv_balance_score']},
        {'item': 'inference_requires_iddx_full', 'value': summary['strict_input_contract']['inference_requires_iddx_full']},
    ]
)
leakage_controls_df = pd.DataFrame(
    [{'control': key, 'value': value} for key, value in summary['leakage_controls'].items()]
)

display(Markdown('**표 6-a. export summary overview**'))
display(summary_overview_df)
display(Markdown('**표 6-b. leakage controls recorded by CLI**'))
display(leakage_controls_df)
